# Image Captioning with Salesforce BLIP

Companion notebook for the To Data & Beyond tutorial. It reproduces the complete conditional and unconditional image-captioning flow with the public `Salesforce/blip-image-captioning-base` checkpoint.

## Important context

The example photographs come from the archived June 2024 tutorial and include conflict imagery. A generated caption is a model prediction, not verified evidence about a person, place, date, or event. Independently verify any factual claim before using it.

In [ ]:
!pip install -q transformers torch pillow requests

In [ ]:
from io import BytesIO

import requests
import torch
from PIL import Image
from transformers import AutoProcessor, BlipForConditionalGeneration

MODEL_ID = "Salesforce/blip-image-captioning-base"
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = BlipForConditionalGeneration.from_pretrained(MODEL_ID).to(device)
model.eval()
print(f"Using {device}")

## Load the archived examples

The repository stores the tutorial images so the notebook runs in Colab without machine-local paths.

In [ ]:
RAW_ASSET_ROOT = (
    "https://raw.githubusercontent.com/To-Data-Beyond/"
    "Generative-AI-Techanical-Tutorials/main/assets"
)
NIGHT_SKY_URL = f"{RAW_ASSET_ROOT}/blip_image_captioning_night_sky.png"
STREET_SCENE_URL = f"{RAW_ASSET_ROOT}/blip_image_captioning_street_scene.png"

def load_image(url: str) -> Image.Image:
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    return Image.open(BytesIO(response.content)).convert("RGB")

night_sky_image = load_image(NIGHT_SKY_URL)
night_sky_image

## Conditional image captioning

Pass both the image and a text prefix to the processor, then decode the generated token IDs.

In [ ]:
text = "a photograph of"
inputs = processor(night_sky_image, text, return_tensors="pt").to(device)

with torch.inference_mode():
    out = model.generate(**inputs, max_new_tokens=40)

conditional_caption = processor.decode(out[0], skip_special_tokens=True)
print(conditional_caption)

## Unconditional image captioning

Omit the text input so BLIP generates a caption from the image alone.

In [ ]:
inputs = processor(night_sky_image, return_tensors="pt").to(device)

with torch.inference_mode():
    out = model.generate(**inputs, max_new_tokens=40)

unconditional_caption = processor.decode(out[0], skip_special_tokens=True)
print(unconditional_caption)

## Try a more specific condition

The next cells reproduce the source article's second comparison. Notice how a prompt can steer the output—and can also encourage the model to invent unsupported specificity.

In [ ]:
street_scene_image = load_image(STREET_SCENE_URL)
street_scene_image

In [ ]:
text = "Israeli soldiers"
inputs = processor(street_scene_image, text, return_tensors="pt").to(device)

with torch.inference_mode():
    out = model.generate(**inputs, max_new_tokens=40)

guided_caption = processor.decode(out[0], skip_special_tokens=True)
print(guided_caption)

In [ ]:
inputs = processor(street_scene_image, return_tensors="pt").to(device)

with torch.inference_mode():
    out = model.generate(**inputs, max_new_tokens=40)

unguided_caption = processor.decode(out[0], skip_special_tokens=True)
print(unguided_caption)

## Optional helper for your own images

Use this function with a Pillow image and an optional text prefix.

In [ ]:
def generate_caption(
    image: Image.Image,
    prompt: str | None = None,
    max_new_tokens: int = 40,
) -> str:
    if prompt:
        inputs = processor(image, prompt, return_tensors="pt").to(device)
    else:
        inputs = processor(image, return_tensors="pt").to(device)

    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens)

    return processor.decode(output[0], skip_special_tokens=True)

print(generate_caption(night_sky_image))